In [4]:
import pandas as pd
import geopandas as gpd

In [5]:
gdf = gpd.read_file("../data/raw/fri-cantons.gpkg")
print(gdf.columns.tolist())
print(gdf.head())


['region_id', 'region_nom', 'prefecture', 'prefectu_1', 'commune_id', 'commune_no', 'canton_id', 'canton_nom', 'canton_n_1', 'grand_lome', 'Duplicata', 'canton_index', 'max_fsi', 'total_pop', 'min_rwi', 'urban_ratio', 'building_count', 'min_dist_basin', 'max_crop_prob', 'norm_fsi', 'norm_pop', 'norm_rwi', 'norm_urban', 'norm_build', 'norm_dist_basin', 'norm_crop', 'FRI', 'geometry']
  region_id region_nom prefecture  prefectu_1 commune_id    commune_no  \
0         A   Maritime        A01  Agoè-Nyivé     A01001  Agoè-Nyivé 1   
1         A   Maritime        A01  Agoè-Nyivé     A01002  Agoè-Nyivé 2   
2         A   Maritime        A01  Agoè-Nyivé     A01003  Agoè-Nyivé 3   
3         A   Maritime        A01  Agoè-Nyivé     A01004  Agoè-Nyivé 4   
4         A   Maritime        A01  Agoè-Nyivé     A01005  Agoè-Nyivé 5   

   canton_id  canton_nom  canton_n_1  grand_lome  ...  max_crop_prob  \
0  A01001001  Agoè-Nyivé  Agoé-Nyivé           1  ...       0.994730   
1  A01002002  Légbassito 

## Etant donné que notre étude se fait principalement ou uniquelent sur Grand-Lomé, on va filtrer notre dataset

In [6]:
lome = gdf[gdf['grand_lome'] == 1][['canton_nom', 'FRI', 'geometry']].copy()
print(len(lome), "cantons dans le Grand-Lomé")
print(lome['FRI'].describe())


13 cantons dans le Grand-Lomé
count    13.000000
mean      0.400712
std       0.148076
min       0.214741
25%       0.300786
50%       0.355862
75%       0.493124
max       0.645001
Name: FRI, dtype: float64


In [8]:
lome['risk_level'] = pd.qcut(lome['FRI'], q=3, labels=['low', 'medium', 'high']) #(IA) Ici on decoupe suivant les trois classes
print(lome[['canton_nom', 'FRI', 'risk_level']].sort_values('FRI'))


       canton_nom       FRI risk_level
24       Amoutivé  0.214741        low
5       Adétikopé  0.222692        low
2      Vakpossito  0.248563        low
22      Bè-Centre  0.300786        low
4        Zanguera  0.307459        low
1      Légbassito  0.343286     medium
25    Aflao-Gakli  0.355862     medium
3      Togblekope  0.414443     medium
0      Agoè-Nyivé  0.493097     medium
23       Bè-Ouest  0.493124       high
27  Aflao-Sagbado  0.536307       high
26        Baguida  0.633900       high
21         Bè-Est  0.645001       high


## Ici on récupère les coordonnées GPS de chaque canton pour pouvoir utiliser **Open-Meteo**

In [9]:
lome['centroid'] = lome.geometry.centroid
lome['lon'] = lome.centroid.to_crs(4326).x
lome['lat'] = lome.centroid.to_crs(4326).y
print(lome[['canton_nom', 'risk_level', 'lat', 'lon']])


       canton_nom risk_level       lat       lon
0      Agoè-Nyivé     medium  6.230589  1.199161
1      Légbassito     medium  6.271143  1.154912
2      Vakpossito        low  6.219399  1.164926
3      Togblekope     medium  6.265311  1.215485
4        Zanguera        low  6.246166  1.121561
5       Adétikopé        low  6.317434  1.224692
21         Bè-Est       high  6.186379  1.269981
22      Bè-Centre        low  6.180691  1.249863
23       Bè-Ouest       high  6.175789  1.220418
24       Amoutivé        low  6.134431  1.213774
25    Aflao-Gakli     medium  6.191449  1.179031
26        Baguida       high  6.182095  1.336767
27  Aflao-Sagbado       high  6.192125  1.122319


In [10]:
lome.drop(columns='centroid').to_file("../data/processed/lome_fri.gpkg", driver="GPKG")
